In [1]:
import sys  
sys.path.insert(1, '../libraries/')

In [2]:
from topology_compiler import *
from droplet_spreading import *

Either run `source ...` before opening the notebook, or define a variable for the GROMACS bin:

```gm_bin = "<path-to-gmx-vib>"```

and pass it to the `topology_compiler` functions.

In [3]:
workdir = os.getcwd()
print("Work directory:",workdir)

Work directory: /home/michele/workflow-refrigerants/example


# Topology generation

In [4]:
help(test_gromacs_availability)

Help on function test_gromacs_availability in module topology_compiler:

test_gromacs_availability(gmx_bin='gmx', oldest_gmx_ver=2024)
    Input:
    - gmx_bin ('gmx'): Path to GROMACS binary file;
    - oldest_gmx_ver (2024): Oldest GROMACS version compatible with the library.



In [5]:
test_gromacs_availability()

In [6]:
help(run_x2top)

Help on function run_x2top in module topology_compiler:

run_x2top(gro_file, ff_folder, name=None, top_file=None, flags='', gmx_bin='gmx')
    Input:
    - gro_file: GROMACS configuration file (.gro) with molecular coordinates;
    - ff_folder: Folder containing interatomic potential definitions;
    - name (None): Name of the molecule in the output .top file (default name: conf. file without extension);
    - top_file (None): Name of the output .top file (default name: conf. file with .top extension);
    - flags (""): Any additional flag to pass to 'gmx x2top';
    - gmx_bin ('gmx'): Path to GROMACS binary file.



In [14]:
run_x2top("HFO-1234zeE.gro", "refrigerants.ff", flags="-alldih -v")
# run_x2top("1233zd-trans.gro", "refrigerants.ff", flags="-alldih")

There are 14 name to type translations in file ./refrigerants.ff

Generating bonds from distances...

There are 5 different atom types in your sample

Generating angles and dihedrals from bonds...

There are   10 proper dihedrals,    0 impropers,   12 angles
            10 pairs,        8 bonds and     9 atoms

Total charge is 0.04289, total mass is 114.043

Topologies generated by gmx x2top can not be trusted at face value. Please verify atomtypes and charges by comparison to other topologies.


                      :-) GROMACS - gmx x2top, 2026.1 (-:

Executable:   /home/michele/gromacs-2026.1/build-gpu/bin/gmx
Data prefix:  /home/michele/gromacs-2026.1/build-gpu
Working dir:  /home/michele/workflow-refrigerants/example
Command line:
  gmx x2top -f HFO-1234zeE.gro -ff refrigerants -name HFO-1234zeE -o HFO-1234zeE.top -alldih -v

Opening force field file ./refrigerants.ff/refrig.n2t
Before cleaning: 10 pairs
Before cleaning: 10 dihedrals

Back Off! I just backed up HFO-1234zeE.top to ./#HFO-1234zeE.top.2#
Topologies generated by gmx x2top can not be trusted at face value. Please verify atomtypes and charges by comparison to other topologies.

GROMACS reminds you: "Three Little Fonzies" (Pulp Fiction)



In [15]:
!cat HFO-1234zeE.top
# !cat 1233zd-trans.top

;
;	File 'HFO-1234zeE.top' was generated
;	By user: michele (1001)
;	On host: denerg33012
;	At date: Fri Jul 10 13:35:00 2026
;
;	This is a include topology file
;
;	Created by:
;	                     :-) GROMACS - gmx x2top, 2026.1 (-:
;	
;	Executable:   /home/michele/gromacs-2026.1/build-gpu/bin/gmx
;	Data prefix:  /home/michele/gromacs-2026.1/build-gpu
;	Working dir:  /home/michele/workflow-refrigerants/example
;	Command line:
;	  gmx x2top -f HFO-1234zeE.gro -ff refrigerants -name HFO-1234zeE -o HFO-1234zeE.top -alldih -v
;	Force field was read from current directory or a relative path - path added.
;

; Include forcefield parameters
#include "./refrigerants.ff/forcefield.itp"

[ moleculetype ]
; Name            nrexcl
HFO-1234zeE         3

[ atoms ]
;   nr       type  resnr residue  atom   cgnr     charge       mass  typeB    chargeB      massB
     1    ref_007      1    UNK    F00      1   -0.19161    18.9984
     2    ref_001      1    UNK    C01      2    0.25325     12.011
 

In [9]:
help(create_itp)

Help on function create_itp in module topology_compiler:

create_itp(top_file, charge_list_file=None, itp_file=None)
    Input:
    - top_file: Topology file (.top) of the molecule;
    - charge_list_file (None): List of partial charges for each atom in the molecule (.txt);
    - itp_file: Output .itp file (default name: .top file with .itp extension).



In [10]:
create_itp("HFO-1234zeE.top", charge_list_file="charges-HFO-1234zeE.txt")

In [11]:
!ls

 box-HFO-1234zeE.gro	      molecular-dynamics-droplet
 charges-HFO-1234zeE.txt      refrigerants.ff
 droplet-evaporation.tar.gz   solvated.gro
 droplet-spreading.gro	      temp-center.gro
 empty.gro		      temp.gro
 hexane.gro		      topology-biphase.top
 hexane.itp		      workflow.ipynb
 HFO-1234zeE-fixed.top	      zirconia-ext.gro
 HFO-1234zeE.gro	      zirconia-ext-shift.gro
 HFO-1234zeE.itp	      zirconia.gro
 HFO-1234zeE.top	      zirconia-header.txt
'#HFO-1234zeE.top.1#'	      zirconia-hexane.gro
'#HFO-1234zeE.top.2#'	      zirconia-HFO-1234zeE.gro
 molecular-dynamics	     '#zirconia-HFO-1234zeE.gro.1#'


In [12]:
help(run_insert_molecules)

Help on function run_insert_molecules in module topology_compiler:

run_insert_molecules(gro_file_f, gro_file_ci, nmol, flags='', gro_file_out=None, gmx_bin='gmx')
    Input:
    - gro_file_f: GROMACS .gro file for the substrate/surface;
    - gro_file_ci: GROMACS .gro file for the solvant;
    - nmol: Number of solvant molecules to insert;
    - flags (""): Any additional flag to pass to "gmx insert-molecules";
    - gro_file_out (None): Output configuration file (default name: combine solvant and substrate with .gro extension)
    - gmx_bin ('gmx'): Path to GROMACS binary file.
    Output:
    - n_added_mol: number of solvane molecules added to the configuration.



In [13]:
n_added_mol = run_insert_molecules("zirconia.gro", "HFO-1234zeE.gro", flags="-try 10 -scale 0.65", nmol=1800)

                 :-) GROMACS - gmx insert-molecules, 2026.1 (-:
Executable:   /home/michele/gromacs-2026.1/build-gpu/bin/gmx
Data prefix:  /home/michele/gromacs-2026.1/build-gpu
Working dir:  /home/michele/workflow-refrigerants/example
Command line:
  gmx insert-molecules -f zirconia.gro -ci HFO-1234zeE.gro -nmol 1800 -o zirconia-HFO-1234zeE.gro -try 10 -scale 0.65
Reading solute configuration
Initialising inter-atomic distances...
Using random seed -154235478
Try 1 success (now 4809 atoms)!
Try 2 success (now 4818 atoms)!
Try 3 success (now 4827 atoms)!
Try 4 success (now 4836 atoms)!
Try 5 success (now 4845 atoms)!
Try 6 success (now 4854 atoms)!
Try 7
Try 8 success (now 4863 atoms)!
Try 9 success (now 4872 atoms)!
Try 10 success (now 4881 atoms)!
Try 11 success (now 4890 atoms)!
Try 12 success (now 4899 atoms)!
Try 13
Try 14 success (now 4908 atoms)!
Try 15 success (now 4917 atoms)!
Try 16 success (now 4926 atoms)!
Try 17 success (now 4935 atoms)!
Try 18 success (now 4944 atoms)!
Tr

Try 1311
Try 1312 success (now 10686 atoms)!
Try 1313
Try 1314
Try 1315 success (now 10695 atoms)!
Try 1316
Try 1317 success (now 10704 atoms)!
Try 1318
Try 1319 success (now 10713 atoms)!
Try 1320
Try 1321
Try 1322 success (now 10722 atoms)!
Try 1323
Try 1324
Try 1325
Try 1326
Try 1327
Try 1328 success (now 10731 atoms)!
Try 1329
Try 1330 success (now 10740 atoms)!
Try 1331
Try 1332
Try 1333
Try 1334
Try 1335
Try 1336
Try 1337
Try 1338
Try 1339
Try 1340
Try 1341
Try 1342
Try 1343
Try 1344 success (now 10749 atoms)!
Try 1345
Try 1346 success (now 10758 atoms)!
Try 1347
Try 1348
Try 1349
Try 1350
Try 1351 success (now 10767 atoms)!
Try 1352 success (now 10776 atoms)!
Try 1353 success (now 10785 atoms)!
Try 1354
Try 1355
Try 1356
Try 1357 success (now 10794 atoms)!
Try 1358
Try 1359
Try 1360 success (now 10803 atoms)!
Try 1361
Try 1362
Try 1363
Try 1364
Try 1365
Try 1366
Try 1367
Try 1368
Try 1369 success (now 10812 atoms)!
Try 1370
Try 1371
Try 1372
Try 1373
Try 1374 success (now 10821 

Try 2699
Try 2700
Try 2701
Try 2702 success (now 13638 atoms)!
Try 2703
Try 2704
Try 2705
Try 2706
Try 2707
Try 2708
Try 2709
Try 2710
Try 2711
Try 2712
Try 2713 success (now 13647 atoms)!
Try 2714
Try 2715 success (now 13656 atoms)!
Try 2716
Try 2717
Try 2718
Try 2719
Try 2720
Try 2721
Try 2722
Try 2723
Try 2724
Try 2725
Try 2726
Try 2727
Try 2728
Try 2729 success (now 13665 atoms)!
Try 2730
Try 2731
Try 2732 success (now 13674 atoms)!
Try 2733
Try 2734 success (now 13683 atoms)!
Try 2735
Try 2736 success (now 13692 atoms)!
Try 2737
Try 2738 success (now 13701 atoms)!
Try 2739
Try 2740 success (now 13710 atoms)!
Try 2741
Try 2742 success (now 13719 atoms)!
Try 2743
Try 2744
Try 2745 success (now 13728 atoms)!
Try 2746
Try 2747 success (now 13737 atoms)!
Try 2748 success (now 13746 atoms)!
Try 2749
Try 2750
Try 2751
Try 2752
Try 2753
Try 2754 success (now 13755 atoms)!
Try 2755
Try 2756
Try 2757
Try 2758 success (now 13764 atoms)!
Try 2759
Try 2760
Try 2761
Try 2762
Try 2763
Try 2764
T

Try 3876
Try 3877
Try 3878
Try 3879
Try 3880
Try 3881
Try 3882
Try 3883
Try 3884 success (now 14988 atoms)!
Try 3885
Try 3886
Try 3887
Try 3888
Try 3889
Try 3890
Try 3891
Try 3892
Try 3893
Try 3894 success (now 14997 atoms)!
Try 3895
Try 3896
Try 3897
Try 3898
Try 3899
Try 3900
Try 3901 success (now 15006 atoms)!
Try 3902
Try 3903
Try 3904 success (now 15015 atoms)!
Try 3905
Try 3906 success (now 15024 atoms)!
Try 3907
Try 3908
Try 3909
Try 3910 success (now 15033 atoms)!
Try 3911
Try 3912
Try 3913
Try 3914
Try 3915
Try 3916 success (now 15042 atoms)!
Try 3917 success (now 15051 atoms)!
Try 3918
Try 3919
Try 3920
Try 3921
Try 3922
Try 3923
Try 3924
Try 3925
Try 3926
Try 3927
Try 3928
Try 3929
Try 3930
Try 3931
Try 3932
Try 3933
Try 3934
Try 3935
Try 3936
Try 3937
Try 3938
Try 3939
Try 3940
Try 3941
Try 3942
Try 3943
Try 3944
Try 3945
Try 3946 success (now 15060 atoms)!
Try 3947
Try 3948 success (now 15069 atoms)!
Try 3949
Try 3950
Try 3951
Try 3952
Try 3953
Try 3954 success (now 15078 

Try 4973
Try 4974
Try 4975
Try 4976
Try 4977 success (now 15897 atoms)!
Try 4978
Try 4979
Try 4980
Try 4981
Try 4982
Try 4983
Try 4984
Try 4985
Try 4986
Try 4987
Try 4988
Try 4989
Try 4990
Try 4991
Try 4992
Try 4993
Try 4994
Try 4995
Try 4996
Try 4997
Try 4998
Try 4999
Try 5000
Try 5001
Try 5002
Try 5003
Try 5004
Try 5005
Try 5006
Try 5007
Try 5008
Try 5009
Try 5010
Try 5011 success (now 15906 atoms)!
Try 5012
Try 5013
Try 5014
Try 5015
Try 5016
Try 5017
Try 5018
Try 5019
Try 5020
Try 5021
Try 5022 success (now 15915 atoms)!
Try 5023
Try 5024
Try 5025
Try 5026
Try 5027
Try 5028
Try 5029
Try 5030
Try 5031 success (now 15924 atoms)!
Try 5032
Try 5033 success (now 15933 atoms)!
Try 5034
Try 5035
Try 5036
Try 5037
Try 5038
Try 5039
Try 5040 success (now 15942 atoms)!
Try 5041
Try 5042
Try 5043
Try 5044
Try 5045
Try 5046 success (now 15951 atoms)!
Try 5047
Try 5048
Try 5049
Try 5050
Try 5051
Try 5052
Try 5053
Try 5054 success (now 15960 atoms)!
Try 5055
Try 5056
Try 5057
Try 5058
Try 5059
T

...

Try 14563
Try 14564
Try 14565
Try 14566 success (now 19821 atoms)!
Try 14567
Try 14568
Try 14569
Try 14570
Try 14571
Try 14572
Try 14573
Try 14574
Try 14575
Try 14576
Try 14577
Try 14578
Try 14579
Try 14580
Try 14581
Try 14582
Try 14583
Try 14584
Try 14585
Try 14586
Try 14587
Try 14588
Try 14589
Try 14590
Try 14591
Try 14592
Try 14593
Try 14594
Try 14595
Try 14596
Try 14597
Try 14598
Try 14599
Try 14600
Try 14601
Try 14602
Try 14603
Try 14604
Try 14605
Try 14606
Try 14607
Try 14608
Try 14609
Try 14610
Try 14611
Try 14612
Try 14613
Try 14614
Try 14615
Try 14616
Try 14617
Try 14618
Try 14619
Try 14620
Try 14621
Try 14622
Try 14623
Try 14624
Try 14625
Try 14626
Try 14627
Try 14628
Try 14629 success (now 19830 atoms)!
Try 14630
Try 14631
Try 14632
Try 14633
Try 14634
Try 14635
Try 14636
Try 14637
Try 14638
Try 14639
Try 14640
Try 14641
Try 14642
Try 14643 success (now 19839 atoms)!
Try 14644
Try 14645
Try 14646
Try 14647
Try 14648
Try 14649
Try 14650
Try 14651
Try 14652
Try 14653
Try 14654

Try 15439
Try 15440
Try 15441
Try 15442
Try 15443
Try 15444
Try 15445
Try 15446
Try 15447
Try 15448
Try 15449
Try 15450
Try 15451
Try 15452
Try 15453
Try 15454
Try 15455
Try 15456
Try 15457
Try 15458
Try 15459
Try 15460
Try 15461
Try 15462
Try 15463
Try 15464
Try 15465
Try 15466
Try 15467
Try 15468
Try 15469
Try 15470
Try 15471
Try 15472
Try 15473
Try 15474
Try 15475
Try 15476
Try 15477
Try 15478
Try 15479
Try 15480
Try 15481
Try 15482
Try 15483
Try 15484
Try 15485
Try 15486
Try 15487
Try 15488
Try 15489
Try 15490
Try 15491
Try 15492
Try 15493
Try 15494
Try 15495
Try 15496
Try 15497
Try 15498
Try 15499
Try 15500
Try 15501
Try 15502
Try 15503
Try 15504
Try 15505
Try 15506
Try 15507
Try 15508
Try 15509
Try 15510
Try 15511
Try 15512
Try 15513
Try 15514
Try 15515 success (now 19983 atoms)!
Try 15516
Try 15517
Try 15518
Try 15519
Try 15520
Try 15521
Try 15522
Try 15523
Try 15524
Try 15525
Try 15526
Try 15527
Try 15528
Try 15529
Try 15530
Try 15531
Try 15532
Try 15533
Try 15534
Try 15535
Try

Try 16303
Try 16304
Try 16305
Try 16306
Try 16307
Try 16308
Try 16309
Try 16310
Try 16311
Try 16312
Try 16313
Try 16314
Try 16315
Try 16316
Try 16317
Try 16318
Try 16319
Try 16320
Try 16321
Try 16322
Try 16323
Try 16324
Try 16325
Try 16326
Try 16327
Try 16328
Try 16329
Try 16330
Try 16331
Try 16332
Try 16333
Try 16334
Try 16335
Try 16336
Try 16337
Try 16338
Try 16339
Try 16340
Try 16341
Try 16342
Try 16343
Try 16344
Try 16345
Try 16346
Try 16347
Try 16348
Try 16349
Try 16350
Try 16351
Try 16352
Try 16353
Try 16354
Try 16355
Try 16356
Try 16357
Try 16358
Try 16359
Try 16360
Try 16361
Try 16362
Try 16363
Try 16364
Try 16365
Try 16366
Try 16367
Try 16368
Try 16369
Try 16370
Try 16371
Try 16372
Try 16373
Try 16374
Try 16375
Try 16376
Try 16377
Try 16378
Try 16379
Try 16380
Try 16381
Try 16382
Try 16383
Try 16384
Try 16385
Try 16386
Try 16387
Try 16388
Try 16389
Try 16390
Try 16391
Try 16392
Try 16393
Try 16394
Try 16395
Try 16396
Try 16397
Try 16398
Try 16399
Try 16400
Try 16401
Try 16402


Try 17175
Try 17176
Try 17177
Try 17178
Try 17179
Try 17180
Try 17181
Try 17182
Try 17183
Try 17184
Try 17185
Try 17186
Try 17187
Try 17188
Try 17189
Try 17190
Try 17191
Try 17192
Try 17193
Try 17194
Try 17195
Try 17196
Try 17197
Try 17198
Try 17199
Try 17200
Try 17201
Try 17202
Try 17203
Try 17204
Try 17205
Try 17206
Try 17207
Try 17208
Try 17209
Try 17210
Try 17211
Try 17212
Try 17213
Try 17214
Try 17215
Try 17216
Try 17217
Try 17218
Try 17219
Try 17220
Try 17221
Try 17222
Try 17223
Try 17224
Try 17225
Try 17226
Try 17227
Try 17228
Try 17229
Try 17230
Try 17231
Try 17232
Try 17233
Try 17234
Try 17235
Try 17236
Try 17237
Try 17238
Try 17239
Try 17240
Try 17241
Try 17242
Try 17243
Try 17244
Try 17245
Try 17246
Try 17247
Try 17248
Try 17249
Try 17250
Try 17251
Try 17252
Try 17253
Try 17254
Try 17255
Try 17256
Try 17257
Try 17258
Try 17259
Try 17260
Try 17261
Try 17262
Try 17263
Try 17264
Try 17265
Try 17266
Try 17267
Try 17268
Try 17269
Try 17270
Try 17271
Try 17272
Try 17273
Try 17274


Output configuration contains 20379 atoms in 3331 residues
GROMACS reminds you: "Computer dating is fine, if you are a computer." (Rita May Brown)


In [14]:
# Make sure it matches the output of 'run_insert_molecules'
print(n_added_mol)

1731


In [15]:
# Divide the number of atoms by 3 (ZrO2)
# 4800->1600
!cat zirconia.gro

GROtesk MACabre and Sinister
 4800
    1ZrO2    ZR    1   0.023   0.145   6.647
    1ZrO2    ZO    2   0.095   0.032   6.312
    1ZrO2    ZO    3   0.129   0.286   6.485
    2ZrO2    ZR    4   0.240   0.145   6.387
    2ZrO2    ZO    5   0.135   0.286   6.744
    2ZrO2    ZO    6   0.169   0.032   6.571
    3ZrO2    ZR    7   0.287   0.379   6.605
    3ZrO2    ZO    8   0.358   0.491   6.420
    3ZrO2    ZO    9   0.392   0.237   6.247
    4ZrO2    ZR   10   0.503   0.379   6.345
    4ZrO2    ZO   11   0.398   0.237   6.507
    4ZrO2    ZO   12   0.432   0.491   6.680
    5ZrO2    ZR   13   0.550   0.145   6.647
    5ZrO2    ZO   14   0.622   0.032   6.312
    5ZrO2    ZO   15   0.655   0.286   6.485
    6ZrO2    ZR   16   0.767   0.145   6.387
    6ZrO2    ZO   17   0.661   0.286   6.744
    6ZrO2    ZO   18   0.695   0.032   6.571
    7ZrO2    ZR   19   0.813   0.379   6.605
    7ZrO2    ZO   20   0.885   0.491   6.420
    7ZrO2    ZO   21   0.919   0.237   6.247
    8ZrO2    ZR   22

In [16]:
help(compile_topology)

Help on function compile_topology in module topology_compiler:

compile_topology(n_sol, n_sub, ff_itp, sol_itp, sub_snippet, output_top='topology-biphase.top', system_name='Biphase')
    Input:
    - n_sol: No. of solvant molecules;
    - n_sub: No. of substrate/surface molecules;
    - ff_itp: Path to ff file (header with [ defaults ] definition);
    - sol_itp: Topology of solvant molecule;
    - sub_snippet: Header of substrate topology (e.g. [ atomtypes ], ..., [ position_restraints ], ...);
    - output_top ("topology-biphase.top"): ...;
    - system_name ("Biphase"): ....



In [17]:
compile_topology(n_added_mol, 1600, "refrigerants.ff/forcefield.itp", "HFO-1234zeE.itp", "zirconia-header.txt")

In [18]:
!cat topology-biphase.top

#include "/home/michele/workflow-refrigerants/example/refrigerants.ff/forcefield.itp" 

[ atomtypes ]
; name  at.num   mass     charge  ptype    sigma       epsilon
; Zirconium atoms (A, original)
  ZR    40       91.22    0.7952  A        0.6227      0.00012
; Oxygen atoms (A)
  ZO  	8        15.9994 -0.3976  A        0.2653      1.16516

[ nonbond_params ]
ZR	ZO	  1	  0.0	     0.0
ZO	ZO	  1	  0.0	     0.0
ZR	ZR	  1	  0.0	     0.0

[ moleculetype ]
; Name        nrexcl
ZrO2          0

[ atoms ]
; id  attype resnr  residue  atname  cgnr   charge      mass      
  1   ZR     1      ZrO2     ZR      1      0.7952      91.22     
  2   ZO     1      ZrO2     ZO      1     -0.3976      15.9994   
  3   ZO     1      ZrO2     ZO      1     -0.3976      15.9994

[ position_restraints ]
  1  1  100000  100000  100000
  2  1  100000  100000  100000
  3  1  100000  100000  100000
  
#include "/home/michele/workflow-refrigerants/example/HFO-1234zeE.itp" 

[ system ]
Biphase

[ molecules ]
ZrO2 

In [19]:
!vmd zirconia-HFO-1234zeE.gro

/usr/local/lib/vmd/vmd_LINUXAMD64: /lib/x86_64-linux-gnu/libGL.so.1: no version information available (required by /usr/local/lib/vmd/vmd_LINUXAMD64)
Info) VMD for LINUXAMD64, version 1.9.3 (November 30, 2016)
Info) http://www.ks.uiuc.edu/Research/vmd/                         
Info) Email questions and bug reports to vmd@ks.uiuc.edu           
Info) Please include this reference in published work using VMD:   
Info)    Humphrey, W., Dalke, A. and Schulten, K., `VMD - Visual   
Info)    Molecular Dynamics', J. Molec. Graphics 1996, 14.1, 33-38.
Info) -------------------------------------------------------------
Info) Multithreading available, 112 CPUs detected.
Info)   CPU features: SSE2 AVX AVX2 FMA KNL:AVX-512F+CD+ER+PF 
Info) Free system memory: 116GB (92%)
Info) Creating CUDA device pool and initializing hardware...
Info) Detected 1 available CUDA accelerator:
Info) [0] NVIDIA RTX A4000   48 SM_8.6 @ 1.56 GHz, 16GB RAM, KTO, AE2, ZCP
Warning) Detected X11 'Composite' extension: if i

In [20]:
!tail zirconia-HFO-1234zeE.gro

 3331UNK    F0020371   5.143   0.960   9.378
 3331UNK    C0120372   5.140   0.962   9.244
 3331UNK    H0220373   5.044   0.928   9.206
 3331UNK    C0320374   5.245   1.002   9.173
 3331UNK    C0420375   5.249   1.006   9.025
 3331UNK    F0520376   5.368   1.050   8.979
 3331UNK    F0620377   5.228   0.884   8.971
 3331UNK    F0720378   5.153   1.089   8.975
 3331UNK    H0820379   5.336   1.034   9.224
   5.26800   5.23400  14.55090


In [4]:
zlow = 3.0
zupp = 14.55090-3.0
carve_condition = lambda x, y, z : (z>zlow)*(z<zupp)
carve_gro("zirconia-HFO-1234zeE.gro",9,carve_condition,mol_type='UNK',output_file="zirconia-HFO-1234zeE-carved.gro")

12261


In [5]:
!gmx editconf -f zirconia-HFO-1234zeE-carved.gro -o zirconia-HFO-1234zeE-carved.pdb

                     :-) GROMACS - gmx editconf, 2026.1 (-:

Executable:   /home/michele/gromacs-2026.1/build-gpu/bin/gmx
Data prefix:  /home/michele/gromacs-2026.1/build-gpu
Working dir:  /home/michele/workflow-refrigerants/example
Command line:
  gmx editconf -f zirconia-HFO-1234zeE-carved.gro -o zirconia-HFO-1234zeE-carved.pdb

Note that major changes are planned in future for editconf, to improve usability and utility.
Read 12261 atoms
Volume: 401.208 nm^3, corresponds to roughly 180500 electrons
No velocities found

GROMACS reminds you: "The time for theory is over" (J. Hajdu)



In [6]:
!cat zirconia-HFO-1234zeE-carved.gro

GROtesk MACabre and Sinister
12261
    1ZrO2    ZR    1   0.023   0.145   6.647
    1ZrO2    ZO    2   0.095   0.032   6.312
    1ZrO2    ZO    3   0.129   0.286   6.485
    2ZrO2    ZR    4   0.240   0.145   6.387
    2ZrO2    ZO    5   0.135   0.286   6.744
    2ZrO2    ZO    6   0.169   0.032   6.571
    3ZrO2    ZR    7   0.287   0.379   6.605
    3ZrO2    ZO    8   0.358   0.491   6.420
    3ZrO2    ZO    9   0.392   0.237   6.247
    4ZrO2    ZR   10   0.503   0.379   6.345
    4ZrO2    ZO   11   0.398   0.237   6.507
    4ZrO2    ZO   12   0.432   0.491   6.680
    5ZrO2    ZR   13   0.550   0.145   6.647
    5ZrO2    ZO   14   0.622   0.032   6.312
    5ZrO2    ZO   15   0.655   0.286   6.485
    6ZrO2    ZR   16   0.767   0.145   6.387
    6ZrO2    ZO   17   0.661   0.286   6.744
    6ZrO2    ZO   18   0.695   0.032   6.571
    7ZrO2    ZR   19   0.813   0.379   6.605
    7ZrO2    ZO   20   0.885   0.491   6.420
    7ZrO2    ZO   21   0.919   0.237   6.247
    8ZrO2    ZR   22

In [7]:
!vmd zirconia-HFO-1234zeE-carved.gro

/usr/local/lib/vmd/vmd_LINUXAMD64: /lib/x86_64-linux-gnu/libGL.so.1: no version information available (required by /usr/local/lib/vmd/vmd_LINUXAMD64)
Info) VMD for LINUXAMD64, version 1.9.3 (November 30, 2016)
Info) http://www.ks.uiuc.edu/Research/vmd/                         
Info) Email questions and bug reports to vmd@ks.uiuc.edu           
Info) Please include this reference in published work using VMD:   
Info)    Humphrey, W., Dalke, A. and Schulten, K., `VMD - Visual   
Info)    Molecular Dynamics', J. Molec. Graphics 1996, 14.1, 33-38.
Info) -------------------------------------------------------------
Info) Multithreading available, 112 CPUs detected.
Info)   CPU features: SSE2 AVX AVX2 FMA KNL:AVX-512F+CD+ER+PF 
Info) Free system memory: 116GB (92%)
Info) Creating CUDA device pool and initializing hardware...
Info) Detected 1 available CUDA accelerator:
Info) [0] NVIDIA RTX A4000   48 SM_8.6 @ 1.56 GHz, 16GB RAM, KTO, AE2, ZCP
Warning) Detected X11 'Composite' extension: if i

# Molecular dynamics

In [8]:
%cd {workdir}/molecular-dynamics/
!ls

/home/michele/workflow-refrigerants/example/molecular-dynamics
mdout.mdp  npt.log	 npt.xtc  nvt.log	sd.mdp	   steep.trr
npt.cpt    npt.mdp	 nvt.cpt  nvt.mdp	steep.edr  system-npt.tpr
npt.edr    npt_prev.cpt  nvt.edr  nvt_prev.cpt	steep.gro  system-nvt.tpr
npt.gro    npt.trr	 nvt.gro  nvt.trr	steep.log  system-sd.tpr


In [9]:
!gmx grompp -p ../topology-triphase.top -c ../zirconia-HFO-1234zeE-carved.gro -r ../zirconia-HFO-1234zeE-carved.gro -f sd.mdp -o system-sd.tpr

                      :-) GROMACS - gmx grompp, 2026.1 (-:

Executable:   /home/michele/gromacs-2026.1/build-gpu/bin/gmx
Data prefix:  /home/michele/gromacs-2026.1/build-gpu
Working dir:  /home/michele/workflow-refrigerants/example/molecular-dynamics
Command line:
  gmx grompp -p ../topology-triphase.top -c ../zirconia-HFO-1234zeE-carved.gro -r ../zirconia-HFO-1234zeE-carved.gro -f sd.mdp -o system-sd.tpr

Setting the LD random seed to -67371665

Generated 187 of the 190 non-bonded parameter combinations
Generating 1-4 interactions: fudge = 0.5

Generated 190 of the 190 1-4 parameter combinations

Excluding 0 bonded neighbours molecule type 'ZrO2'

turning H bonds into constraints...

Excluding 3 bonded neighbours molecule type 'HFO-1234zeE'

turning H bonds into constraints...

NOTE 1 [file topology-triphase.top, line 37]:
  In moleculetype 'ZrO2' 3 atoms are not bound by a potential or constraint
  to any other atom in the same moleculetype. Although technically this
  might not caus

In [10]:
!gmx mdrun -v -s system-sd.tpr -deffnm steep

                      :-) GROMACS - gmx mdrun, 2026.1 (-:

Executable:   /home/michele/gromacs-2026.1/build-gpu/bin/gmx
Data prefix:  /home/michele/gromacs-2026.1/build-gpu
Working dir:  /home/michele/workflow-refrigerants/example/molecular-dynamics
Command line:
  gmx mdrun -v -s system-sd.tpr -deffnm steep


Back Off! I just backed up steep.log to ./#steep.log.1#
Reading file system-sd.tpr, VERSION 2026.1 (single precision)
Update groups can not be used for this system because atoms that are (in)directly constrained together are interdispersed with other atoms

GPU halo exchange will not be activated because:
  Energy minimization is not supported.

On host denerg33012 1 GPU selected for this run.
Mapping of GPU IDs to the 4 GPU tasks in the 4 ranks on this node:
  PP:0,PP:0,PP:0,PP:0
PP tasks will do non-perturbed short-ranged interactions on the GPU
PP task will update and constrain coordinates on the CPU
Using 4 MPI threads
Using 28 OpenMP threads per tMPI thread


Back Off! I jus

Step=  127, Dmax= 2.1e-03 nm, Epot= -9.17193e+05 Fmax= 1.07995e+03, atom= 10573
Step=  129, Dmax= 1.3e-03 nm, Epot= -9.17218e+05 Fmax= 5.49199e+02, atom= 10573
Step=  130, Dmax= 1.5e-03 nm, Epot= -9.17238e+05 Fmax= 1.51609e+03, atom= 10573
Step=  131, Dmax= 1.9e-03 nm, Epot= -9.17270e+05 Fmax= 8.41950e+02, atom= 10573
Step=  132, Dmax= 2.2e-03 nm, Epot= -9.17271e+05 Fmax= 2.09417e+03, atom= 10573
Step=  133, Dmax= 2.7e-03 nm, Epot= -9.17313e+05 Fmax= 1.31261e+03, atom= 10573
Step=  135, Dmax= 1.6e-03 nm, Epot= -9.17341e+05 Fmax= 7.85288e+02, atom= 10573
Step=  136, Dmax= 1.9e-03 nm, Epot= -9.17352e+05 Fmax= 1.63260e+03, atom= 10573
Step=  137, Dmax= 2.3e-03 nm, Epot= -9.17379e+05 Fmax= 1.37979e+03, atom= 10573
Step=  139, Dmax= 1.4e-03 nm, Epot= -9.17409e+05 Fmax= 3.78494e+02, atom= 10573
Step=  140, Dmax= 1.7e-03 nm, Epot= -9.17434e+05 Fmax= 1.86676e+03, atom= 10573
Step=  141, Dmax= 2.0e-03 nm, Epot= -9.17475e+05 Fmax= 6.72354e+02, atom= 10573
Step=  143, Dmax= 1.2e-03 nm, Epot= -9.1

Step=  276, Dmax= 2.2e-03 nm, Epot= -9.19275e+05 Fmax= 1.58450e+03, atom= 7405
Step=  277, Dmax= 2.6e-03 nm, Epot= -9.19282e+05 Fmax= 1.85586e+03, atom= 7405
Step=  278, Dmax= 3.2e-03 nm, Epot= -9.19287e+05 Fmax= 2.22781e+03, atom= 7405
Step=  279, Dmax= 3.8e-03 nm, Epot= -9.19291e+05 Fmax= 2.71848e+03, atom= 7405
Step=  280, Dmax= 4.6e-03 nm, Epot= -9.19294e+05 Fmax= 3.16948e+03, atom= 7405
Step=  282, Dmax= 2.7e-03 nm, Epot= -9.19324e+05 Fmax= 3.92539e+02, atom= 7405
Step=  283, Dmax= 3.3e-03 nm, Epot= -9.19335e+05 Fmax= 3.78955e+03, atom= 7405
Step=  284, Dmax= 3.9e-03 nm, Epot= -9.19371e+05 Fmax= 1.34546e+03, atom= 7405
Step=  286, Dmax= 2.4e-03 nm, Epot= -9.19378e+05 Fmax= 1.69557e+03, atom= 7405
Step=  287, Dmax= 2.8e-03 nm, Epot= -9.19385e+05 Fmax= 1.99980e+03, atom= 7405
Step=  288, Dmax= 3.4e-03 nm, Epot= -9.19390e+05 Fmax= 2.39112e+03, atom= 7405
Step=  289, Dmax= 4.1e-03 nm, Epot= -9.19393e+05 Fmax= 2.92232e+03, atom= 7405
Step=  290, Dmax= 4.9e-03 nm, Epot= -9.19394e+05 Fma

Step=  447, Dmax= 1.6e-03 nm, Epot= -9.20454e+05 Fmax= 3.78873e+02, atom= 7405
Step=  448, Dmax= 1.9e-03 nm, Epot= -9.20461e+05 Fmax= 2.01313e+03, atom= 7405
Step=  449, Dmax= 2.3e-03 nm, Epot= -9.20472e+05 Fmax= 9.10454e+02, atom= 7405
Step=  451, Dmax= 1.4e-03 nm, Epot= -9.20477e+05 Fmax= 8.23793e+02, atom= 7405
Step=  452, Dmax= 1.6e-03 nm, Epot= -9.20480e+05 Fmax= 1.28650e+03, atom= 7405
Step=  453, Dmax= 1.9e-03 nm, Epot= -9.20485e+05 Fmax= 1.21558e+03, atom= 7405
Step=  454, Dmax= 2.3e-03 nm, Epot= -9.20487e+05 Fmax= 1.81607e+03, atom= 7405
Step=  455, Dmax= 2.8e-03 nm, Epot= -9.20492e+05 Fmax= 1.79095e+03, atom= 7405
Step=  457, Dmax= 1.7e-03 nm, Epot= -9.20502e+05 Fmax= 3.90503e+02, atom= 7405
Step=  458, Dmax= 2.0e-03 nm, Epot= -9.20506e+05 Fmax= 2.18226e+03, atom= 7405
Step=  459, Dmax= 2.4e-03 nm, Epot= -9.20518e+05 Fmax= 9.59276e+02, atom= 7405
Step=  461, Dmax= 1.5e-03 nm, Epot= -9.20523e+05 Fmax= 9.05476e+02, atom= 7405
Step=  462, Dmax= 1.7e-03 nm, Epot= -9.20526e+05 Fma

Step=  577, Dmax= 1.7e-03 nm, Epot= -9.20996e+05 Fmax= 1.72484e+03, atom= 10591
Step=  578, Dmax= 2.0e-03 nm, Epot= -9.21004e+05 Fmax= 8.75833e+02, atom= 10591
Step=  580, Dmax= 1.2e-03 nm, Epot= -9.21008e+05 Fmax= 6.63254e+02, atom= 10591
Step=  581, Dmax= 1.4e-03 nm, Epot= -9.21010e+05 Fmax= 1.21830e+03, atom= 10591
Step=  582, Dmax= 1.7e-03 nm, Epot= -9.21014e+05 Fmax= 1.00281e+03, atom= 10591
Step=  584, Dmax= 1.0e-03 nm, Epot= -9.21019e+05 Fmax= 3.48831e+02, atom= 10591
Step=  585, Dmax= 1.2e-03 nm, Epot= -9.21023e+05 Fmax= 1.22826e+03, atom= 10591
Step=  586, Dmax= 1.5e-03 nm, Epot= -9.21029e+05 Fmax= 7.15398e+02, atom= 10591
Step=  587, Dmax= 1.8e-03 nm, Epot= -9.21029e+05 Fmax= 1.57839e+03, atom= 10591
Step=  588, Dmax= 2.2e-03 nm, Epot= -9.21035e+05 Fmax= 1.21619e+03, atom= 10591
Step=  590, Dmax= 1.3e-03 nm, Epot= -9.21041e+05 Fmax= 4.41586e+02, atom= 10591
Step=  591, Dmax= 1.6e-03 nm, Epot= -9.21042e+05 Fmax= 1.59016e+03, atom= 10591
Step=  592, Dmax= 1.9e-03 nm, Epot= -9.2

...

Step= 5863, Dmax= 6.1e-04 nm, Epot= -9.25386e+05 Fmax= 1.12974e+02, atom= 8818
Step= 5864, Dmax= 7.3e-04 nm, Epot= -9.25386e+05 Fmax= 8.10885e+02, atom= 8818
Step= 5865, Dmax= 8.7e-04 nm, Epot= -9.25387e+05 Fmax= 3.22522e+02, atom= 8818
Step= 5867, Dmax= 5.2e-04 nm, Epot= -9.25387e+05 Fmax= 3.50389e+02, atom= 8818
Step= 5868, Dmax= 6.3e-04 nm, Epot= -9.25388e+05 Fmax= 4.68017e+02, atom= 8818
Step= 5869, Dmax= 7.5e-04 nm, Epot= -9.25388e+05 Fmax= 5.03011e+02, atom= 8818
Step= 5870, Dmax= 9.0e-04 nm, Epot= -9.25388e+05 Fmax= 6.73093e+02, atom= 8818
Step= 5871, Dmax= 1.1e-03 nm, Epot= -9.25388e+05 Fmax= 7.27474e+02, atom= 8818
Step= 5873, Dmax= 6.5e-04 nm, Epot= -9.25389e+05 Fmax= 1.18645e+02, atom= 8818
Step= 5874, Dmax= 7.8e-04 nm, Epot= -9.25390e+05 Fmax= 8.75792e+02, atom= 8818
Step= 5875, Dmax= 9.4e-04 nm, Epot= -9.25391e+05 Fmax= 3.42459e+02, atom= 8818
Step= 5877, Dmax= 5.6e-04 nm, Epot= -9.25391e+05 Fmax= 3.81121e+02, atom= 8818
Step= 5878, Dmax= 6.7e-04 nm, Epot= -9.25391e+05 Fma

Step= 6022, Dmax= 6.7e-04 nm, Epot= -9.25451e+05 Fmax= 2.67290e+02, atom= 8818
Step= 6024, Dmax= 4.0e-04 nm, Epot= -9.25452e+05 Fmax= 2.47902e+02, atom= 8818
Step= 6025, Dmax= 4.8e-04 nm, Epot= -9.25452e+05 Fmax= 3.81205e+02, atom= 8818
Step= 6026, Dmax= 5.8e-04 nm, Epot= -9.25452e+05 Fmax= 3.62311e+02, atom= 8818
Step= 6027, Dmax= 6.9e-04 nm, Epot= -9.25452e+05 Fmax= 5.40843e+02, atom= 8818
Step= 6028, Dmax= 8.3e-04 nm, Epot= -9.25453e+05 Fmax= 5.31359e+02, atom= 8818
Step= 6030, Dmax= 5.0e-04 nm, Epot= -9.25453e+05 Fmax= 1.17852e+02, atom= 8818
Step= 6031, Dmax= 6.0e-04 nm, Epot= -9.25454e+05 Fmax= 6.43281e+02, atom= 8818
Step= 6032, Dmax= 7.2e-04 nm, Epot= -9.25454e+05 Fmax= 2.90807e+02, atom= 8818
Step= 6034, Dmax= 4.3e-04 nm, Epot= -9.25455e+05 Fmax= 2.63398e+02, atom= 8818
Step= 6035, Dmax= 5.2e-04 nm, Epot= -9.25455e+05 Fmax= 4.12185e+02, atom= 8818
Step= 6036, Dmax= 6.2e-04 nm, Epot= -9.25455e+05 Fmax= 3.87244e+02, atom= 8818
Step= 6037, Dmax= 7.4e-04 nm, Epot= -9.25456e+05 Fma

Step= 6174, Dmax= 1.2e-03 nm, Epot= -9.25500e+05 Fmax= 3.81422e+02, atom= 8818
Step= 6176, Dmax= 7.1e-04 nm, Epot= -9.25501e+05 Fmax= 5.45542e+02, atom= 8818
Step= 6178, Dmax= 4.3e-04 nm, Epot= -9.25501e+05 Fmax= 2.26050e+01, atom= 8175
Step= 6179, Dmax= 5.1e-04 nm, Epot= -9.25505e+05 Fmax= 1.99914e+02, atom= 8818
Step= 6180, Dmax= 6.1e-04 nm, Epot= -9.25506e+05 Fmax= 5.87044e+02, atom= 8818
Step= 6181, Dmax= 7.4e-04 nm, Epot= -9.25506e+05 Fmax= 3.72366e+02, atom= 8818
Step= 6183, Dmax= 4.4e-04 nm, Epot= -9.25506e+05 Fmax= 1.97392e+02, atom= 8818
Step= 6184, Dmax= 5.3e-04 nm, Epot= -9.25507e+05 Fmax= 4.98156e+02, atom= 8818
Step= 6185, Dmax= 6.4e-04 nm, Epot= -9.25507e+05 Fmax= 3.23525e+02, atom= 8818
Step= 6186, Dmax= 7.6e-04 nm, Epot= -9.25507e+05 Fmax= 6.73765e+02, atom= 8818
Step= 6187, Dmax= 9.2e-04 nm, Epot= -9.25508e+05 Fmax= 5.10635e+02, atom= 8818
Step= 6189, Dmax= 5.5e-04 nm, Epot= -9.25508e+05 Fmax= 2.05589e+02, atom= 8818
Step= 6190, Dmax= 6.6e-04 nm, Epot= -9.25508e+05 Fma

Step= 6309, Dmax= 1.3e-03 nm, Epot= -9.25548e+05 Fmax= 9.08882e+02, atom= 8818
Step= 6311, Dmax= 7.9e-04 nm, Epot= -9.25548e+05 Fmax= 1.06950e+02, atom= 8818
Step= 6312, Dmax= 9.4e-04 nm, Epot= -9.25549e+05 Fmax= 1.13748e+03, atom= 8818
Step= 6313, Dmax= 1.1e-03 nm, Epot= -9.25550e+05 Fmax= 3.24964e+02, atom= 8818
Step= 6315, Dmax= 6.8e-04 nm, Epot= -9.25550e+05 Fmax= 5.61468e+02, atom= 8818
Step= 6316, Dmax= 8.1e-04 nm, Epot= -9.25551e+05 Fmax= 4.90587e+02, atom= 8818
Step= 6317, Dmax= 9.8e-04 nm, Epot= -9.25551e+05 Fmax= 7.82385e+02, atom= 8818
Step= 6318, Dmax= 1.2e-03 nm, Epot= -9.25551e+05 Fmax= 7.33973e+02, atom= 8818
Step= 6320, Dmax= 7.0e-04 nm, Epot= -9.25552e+05 Fmax= 1.81564e+02, atom= 8818
Step= 6322, Dmax= 4.2e-04 nm, Epot= -9.25552e+05 Fmax= 3.60059e+02, atom= 8818
Step= 6323, Dmax= 5.1e-04 nm, Epot= -9.25552e+05 Fmax= 3.00984e+02, atom= 8818
Step= 6324, Dmax= 6.1e-04 nm, Epot= -9.25553e+05 Fmax= 4.81984e+02, atom= 8818
Step= 6325, Dmax= 7.3e-04 nm, Epot= -9.25553e+05 Fma

Step= 6445, Dmax= 7.2e-04 nm, Epot= -9.25593e+05 Fmax= 5.12049e+02, atom= 8818
Step= 6446, Dmax= 8.7e-04 nm, Epot= -9.25593e+05 Fmax= 6.18745e+02, atom= 8818
Step= 6448, Dmax= 5.2e-04 nm, Epot= -9.25594e+05 Fmax= 5.42812e+01, atom= 8818
Step= 6449, Dmax= 6.3e-04 nm, Epot= -9.25595e+05 Fmax= 7.86489e+02, atom= 8818
Step= 6450, Dmax= 7.5e-04 nm, Epot= -9.25596e+05 Fmax= 1.83161e+02, atom= 8818
Step= 6452, Dmax= 4.5e-04 nm, Epot= -9.25596e+05 Fmax= 4.07648e+02, atom= 8818
Step= 6453, Dmax= 5.4e-04 nm, Epot= -9.25596e+05 Fmax= 2.88873e+02, atom= 8818
Step= 6454, Dmax= 6.5e-04 nm, Epot= -9.25596e+05 Fmax= 5.57985e+02, atom= 8818
Step= 6455, Dmax= 7.8e-04 nm, Epot= -9.25597e+05 Fmax= 4.46655e+02, atom= 8818
Step= 6456, Dmax= 9.3e-04 nm, Epot= -9.25597e+05 Fmax= 7.69804e+02, atom= 8818
Step= 6457, Dmax= 1.1e-03 nm, Epot= -9.25597e+05 Fmax= 6.78484e+02, atom= 8818
Step= 6459, Dmax= 6.7e-04 nm, Epot= -9.25598e+05 Fmax= 1.96349e+02, atom= 8818
Step= 6461, Dmax= 4.0e-04 nm, Epot= -9.25598e+05 Fma

In [11]:
!ls

 mdout.mdp   npt_prev.cpt   nvt.log	  '#steep.edr.1#'  '#steep.trr.1#'
 npt.cpt     npt.trr	    nvt.mdp	   steep.gro	    system-npt.tpr
 npt.edr     npt.xtc	    nvt_prev.cpt  '#steep.gro.1#'   system-nvt.tpr
 npt.gro     nvt.cpt	    nvt.trr	   steep.log	    system-sd.tpr
 npt.log     nvt.edr	    sd.mdp	  '#steep.log.1#'  '#system-sd.tpr.1#'
 npt.mdp     nvt.gro	    steep.edr	   steep.trr


In [12]:
!gmx grompp -p ../topology-triphase.top -c steep.gro -r ../zirconia-HFO-1234zeE-carved.gro -f nvt.mdp -o system-nvt.tpr

                      :-) GROMACS - gmx grompp, 2026.1 (-:

Executable:   /home/michele/gromacs-2026.1/build-gpu/bin/gmx
Data prefix:  /home/michele/gromacs-2026.1/build-gpu
Working dir:  /home/michele/workflow-refrigerants/example/molecular-dynamics
Command line:
  gmx grompp -p ../topology-triphase.top -c steep.gro -r ../zirconia-HFO-1234zeE-carved.gro -f nvt.mdp -o system-nvt.tpr

Ignoring obsolete mdp entry 'ns-type'
Setting the LD random seed to -1879056417

Generated 187 of the 190 non-bonded parameter combinations
Generating 1-4 interactions: fudge = 0.5

Generated 190 of the 190 1-4 parameter combinations

Excluding 0 bonded neighbours molecule type 'ZrO2'

turning H bonds into constraints...

Excluding 3 bonded neighbours molecule type 'HFO-1234zeE'

turning H bonds into constraints...

Taking velocities from 'steep.gro', all velocities are zero

NOTE 1 [file topology-triphase.top, line 37]:
  In moleculetype 'ZrO2' 3 atoms are not bound by a potential or constraint
  to any o

In [13]:
!gmx mdrun -v -s system-nvt.tpr -deffnm nvt

                      :-) GROMACS - gmx mdrun, 2026.1 (-:

Executable:   /home/michele/gromacs-2026.1/build-gpu/bin/gmx
Data prefix:  /home/michele/gromacs-2026.1/build-gpu
Working dir:  /home/michele/workflow-refrigerants/example/molecular-dynamics
Command line:
  gmx mdrun -v -s system-nvt.tpr -deffnm nvt


Back Off! I just backed up nvt.log to ./#nvt.log.1#
Reading file system-nvt.tpr, VERSION 2026.1 (single precision)
Changing nstlist from 10 to 100, rlist from 1 to 1.07

Update groups can not be used for this system because atoms that are (in)directly constrained together are interdispersed with other atoms

On host denerg33012 1 GPU selected for this run.
Mapping of GPU IDs to the 4 GPU tasks in the 4 ranks on this node:
  PP:0,PP:0,PP:0,PP:0
PP tasks will do non-perturbed short-ranged and most bonded interactions on the GPU
PP task will update and constrain coordinates on the CPU
GPU direct communication will be used between MPI ranks.
Using 4 MPI threads
Using 28 OpenMP threads

step 55400, remaining wall clock time:    38 s          imb F  0% ol 0.98  imb F  3% vol 0.96  imb F  5% vol 0.94  imb F  1% vol 0.93  imb F  3% vol 0.92  imb F 10% vol 0.87  imb F  3% vol 0.86  imb F  6% vol 0.83  imb F  1% vol 0.83  imb F  2% vol 0.83  imb F 13% vol 0.85  imb F  8% vol 0.81  imb F  1% vol 0.81  imb F  8% vol 0.78  imb F  1% vol 0.78  imb F  5% vol 0.77  imb F  1% vol 0.76  imb F  2% vol 0.76  imb F  8% vol 0.73  imb F  4% vol 0.73  imb F  3% vol 0.73  imb F  9% vol 0.70  imb F  2% vol 0.69  imb F  6% vol 0.69  imb F  3% vol 0.69  imb F  3% vol 0.68  imb F  1% vol 0.68  imb F  4% vol 0.68  imb F  3% vol 0.67  imb F 20% vol 0.65! imb F 35% vol 0.65! imb F  1% vol 0.66  imb F 12% vol 0.67  imb F  3% vol 0.66  imb F 11% vol 0.65! imb F  8% vol 0.65! imb F  1% vol 0.65! imb F  1% vol 0.65! imb F  3% vol 0.66  imb F  2% vol 0.66  imb F  4% vol 0.68  imb F  3% vol 0.68  imb F  3% vol 0.68  imb F  1% vol 0.68  imb F  4% vol 0.69  imb F  2% vol 0.68  imb F  5% vol 0.68  imb F

In [14]:
!ls

 mdout.mdp      nvt.cpt        nvt.trr	        steep.trr
 npt.cpt        nvt.edr       '#nvt.trr.1#'    '#steep.trr.1#'
 npt.edr       '#nvt.edr.1#'   sd.mdp	        system-npt.tpr
 npt.gro        nvt.gro        steep.edr        system-nvt.tpr
 npt.log       '#nvt.gro.1#'  '#steep.edr.1#'  '#system-nvt.tpr.1#'
 npt.mdp        nvt.log        steep.gro        system-sd.tpr
 npt_prev.cpt  '#nvt.log.1#'  '#steep.gro.1#'  '#system-sd.tpr.1#'
 npt.trr        nvt.mdp        steep.log
 npt.xtc        nvt_prev.cpt  '#steep.log.1#'


In [16]:
!vmd nvt.trr nvt.gro

/usr/local/lib/vmd/vmd_LINUXAMD64: /lib/x86_64-linux-gnu/libGL.so.1: no version information available (required by /usr/local/lib/vmd/vmd_LINUXAMD64)
Info) VMD for LINUXAMD64, version 1.9.3 (November 30, 2016)
Info) http://www.ks.uiuc.edu/Research/vmd/                         
Info) Email questions and bug reports to vmd@ks.uiuc.edu           
Info) Please include this reference in published work using VMD:   
Info)    Humphrey, W., Dalke, A. and Schulten, K., `VMD - Visual   
Info)    Molecular Dynamics', J. Molec. Graphics 1996, 14.1, 33-38.
Info) -------------------------------------------------------------
Info) Multithreading available, 112 CPUs detected.
Info)   CPU features: SSE2 AVX AVX2 FMA KNL:AVX-512F+CD+ER+PF 
Info) Free system memory: 116GB (92%)
Info) Creating CUDA device pool and initializing hardware...
Info) Detected 1 available CUDA accelerator:
Info) [0] NVIDIA RTX A4000   48 SM_8.6 @ 1.56 GHz, 16GB RAM, KTO, AE2, ZCP
Warning) Detected X11 'Composite' extension: if i

In [ ]:
!gmx grompp -p ../topology-biphase.top -c nvt.gro -r ../zirconia-HFO-1234zeE.gro -f npt.mdp -o system-npt.tpr

In [ ]:
!gmx mdrun -v -s system-npt.tpr -deffnm npt

In [ ]:
!ls

In [ ]:
!vmd npt.xtc npt.gro

# Test with hexane

In [ ]:
%cd {workdir}

In [ ]:
n_added_mol = run_insert_molecules("zirconia.gro", "hexane.gro", flags="-try 10 -scale 0.65", nmol=1000)

In [ ]:
compile_topology(n_added_mol, 1600, "refrigerants.ff/forcefield.itp", "hexane.itp", "zirconia-header.txt")

In [ ]:
%cd {workdir}/molecular-dynamics/
!ls

In [ ]:
!gmx grompp -p ../topology-biphase.top -c ../zirconia-hexane.gro -r ../zirconia-hexane.gro -f sd.mdp -o system-sd.tpr

In [ ]:
!gmx mdrun -v -s system-sd.tpr -deffnm steep

In [ ]:
!gmx grompp -p ../topology-biphase.top -c steep.gro -r ../zirconia-hexane.gro -f nvt.mdp -o system-nvt.tpr

In [ ]:
!gmx mdrun -v -s system-nvt.tpr -deffnm nvt

In [ ]:
!gmx grompp -p ../topology-biphase.top -c nvt.gro -r ../zirconia-hexane.gro -f npt.mdp -o system-npt.tpr

In [ ]:
!gmx mdrun -v -s system-npt.tpr -deffnm npt

In [ ]:
!vmd npt.xtc npt.gro